# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1 : Esteva Matias (E-1253/1)

## 1. Explicacion completa del pipeline

El sistema está compuesto por tres etapas integradas:

**Etapa 1. Búsqueda por similitud:**
Las imágenes del dataset de train se procesan con EfficientNet-B0 preentrenado en ImageNet sin la capa de clasificación final para generar embeddings de 1280 dimensiones. Estos
vectores se indexan en la base vectorial (pgvector sobre PostgreSQL). Con la llegada de una imagen de consulta se genera su embedding y se buscan los top-K vectores más similares por similitud coseno y retorna las raza correspondiente.

**Etapa 2. Clasificación supervisada:**
Se entrena un clasificador de razas sobre los mismos datos. Modelo A: fine-tuning de ResNet18 preentrenado en ImageNet, reemplazando su capa final por una de 70 clases.
Modelo B: CNN propia entrenada desde cero con arquitectura de 5 bloques convolucionales.
Ambos modelos se guardan como checkpoints `.pth` y son cargados por el backend para clasificar imágenes individuales.

**Etapa 3. Detección + clasificación:**
YOLOv8n (preentrenado en COCO, clase 16 = dog) detecta los bounding boxes de todos los perros en la imagen. Cada recorte se pasa al clasificador de Etapa 2, que predice la raza
con su score de confianza. El resultado se muestra como JSON con coordenadas, raza y scores de detección y clasificación.

**Integración:**
El backend FastAPI orquesta los tres servicios (`SimilarityService`, `ClassifierService`,
`DetectionService`). El frontend Gradio expone una pestaña por etapa permitiendo al usuario seleccionar el modelo activo y visualizar los resultados.

## 2. Dataset

El dataset contiene 9346 imágenes de 70 razas de perros, todas en resolución 224×224 píxeles.

| Split | Imágenes | Proporción |
|-------|----------|------------|
| Train | 7946     | ~85%       |
| Valid | 700      | ~7.5%      |
| Test  | 700      | ~7.5%      |

La distribución por clase no es uniforme: las razas mayoritarias (Shih-Tzu, Lhasa) superan las 200 imágenes totales mientras que las minoritarias (American Hairless, Yorkie) apenas alcanzan entre 70 y 90. Este desbalance justifica el uso de `WeightedRandomSampler` y pesos en la función de pérdida durante el entrenamiento.

Los splits train/valid/test están predefinidos por la estructura de carpetas del dataset descargado. El conjunto de test (700 imágenes, 10 por raza) nunca fue visto por los
modelos durante el entrenamiento y se usó exclusivamente para reportar métricas finales.

Como conjunto independiente de evaluación se utilizaron imágenes descargadas de internet (no pertenecientes al dataset original) para probar el pipeline de Etapa 3 en condiciones
con mas de un perro por foto.

## 3. Preprocesamiento

Se definieron dos pipelines de transformaciones:

**Train (data augmentation):**
- `Resize(224×224)`: garantiza tamaño uniforme necesario para imágenes externas.
- `RandomHorizontalFlip(p=0.5)`: la raza no varía con la orientación horizontal.
- `RandomRotation(±15°)`: simula distintos ángulos de toma.
- `ColorJitter(brightness=0.3, contrast=0.3)`: compensa variaciones de iluminación.
- `GaussianBlur(kernel=3)`: simula imágenes levemente desenfocadas.
- `ToTensor` + `Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])`: normalización con estadísticas de ImageNet, requerida por ResNet18 preentrenado.

**Validación y test:**
Solo `Resize`, `ToTensor` y `Normalize`. La evaluación debe reflejar condiciones reales en vez de versiones alteradas artificialmente.

**Balanceo de clases:**
Se combinaron dos estrategias: `WeightedRandomSampler` (sobremuestrea clases minoritarias en cada época) y `CrossEntropyLoss` con pesos inversos a la frecuencia de cada clase.

## 4. Justificacion de los modelos elegidos

**EfficientNet-B0 (Etapa 1 - baseline):**
Seleccionado por su balance entre calidad del embedding (1280 dims, 77.1% Top-1 en ImageNet) y eficiencia computacional (5.3M parámetros). Se usa sin reentrenamiento (solo como extractor de features).

**ResNet18 fine-tuned (Etapa 2 - Modelo A):**
Arquitectura consolidada con conexiones residuales que evitan el vanishing gradient.
Con 11M de parámetros y preentrenamiento en ImageNet, el fine-tuning requiere solo ajustar la capa final (512 → 70 clases). Converge rápido con pocas épocas.
Tiempo de entrenamiento: 11 minutos en GPU.

**CNN custom (Etapa 2 - Modelo B):**
Red propia de 5 bloques convolucionales (3→32→64→128→256→512 canales), entrenada desde cero. Menor accuracy que ResNet18 pero
útil para comparar el impacto del transfer learning. Tiempo: 102 minutos en GPU.

**YOLOv8n (Etapa 3):**
Modelo preentrenado en COCO (80 clases, clase 16 = dog). Se usa sin
reentrenamiento. La variante `nano` fue elegida por su velocidad de inferencia con confidence threshold = 0.4.

## 5. Proceso de entrenamiento e hiperparametros

| Parámetro               | ResNet18 (A)                  | CNN custom (B)                |
|-------------------------|-------------------------------|-------------------------------|
| Image size              | 224×224                       | 224×224                       |
| Batch size              | 128                           | 128                           |
| Epochs                  | 20                            | 100                           |
| Optimizador             | Adam (lr=3e-4)                | Adam (lr=3e-4)                |
| Scheduler               | CosineAnnealingLR (T_max=20)  | CosineAnnealingLR (T_max=100) |
| Loss                    | CrossEntropyLoss (ponderada)  | CrossEntropyLoss (ponderada)  |
| Balanceo                | WeightedRandomSampler         | WeightedRandomSampler         |
| Tiempo entrenamiento    | 11 min (GPU)                  | 102 min (GPU)                 |

El checkpoint se guarda solo cuando `val_acc` mejora respecto al mejor anterior evitando guardar modelos en sobreajuste.

`CosineAnnealingLR` fue elegido sobre `StepLR` porque baja el learning rate suavemente durante todo el entrenamiento sin caídas abruptas que detengan el aprendizaje prematuramente.

## 6. Resultados obtenidos

### Etapa 1 — NDCG@10

| Métrica   | Valor  |
|-----------|--------|
| NDCG@10   | 0.9716 |

NDCG@10 = 0.97 indica que casi todos los top-10 resultados recuperados corresponden
a la misma raza que la imagen de consulta. Un valor cercano a 1.0 refleja que
EfficientNet-B0 genera embeddings suficientemente discriminativos para separar las
70 razas en el espacio vectorial.

### Etapa 2 — Clasificación supervisada

| Métrica      | ResNet18 (A) | CNN custom (B) |
|--------------|-------------|----------------|
| Accuracy     | 0.9543      | 0.8257         |
| Precision    | 0.9584      | 0.8443         |
| Recall       | 0.9543      | 0.8257         |
| Specificity  | 0.9993      | 0.9975         |
| F1-Score     | 0.9538      | 0.8252         |

Las métricas se calcularon sobre el conjunto de test (700 imágenes nunca vistas
durante el entrenamiento).

## 7. Comparación entre enfoques

### Búsqueda por similitud vs clasificación supervisada

La búsqueda por similitud no requiere entrenamiento supervisado: usa EfficientNet-B0 preentrenado, genera embeddings, los almacena, compara por similitud y retorna los K vecinos más cercanos.
Es flexible (no necesita reentrenar al agregar razas nuevas) pero no produce una clase única como salida sino un ranking de resultados. 

La clasificación supervisada (Etapa 2) produce una predicción directa con score de confianza, pero requiere reentrenamiento ante nuevas clases.

### ResNet18 fine-tuned vs CNN custom

ResNet18 supera a la CNN custom en todas las métricas con menos de una décima parte del tiempo de entrenamiento (11 vs 102 minutos). La diferencia principal es el
transfer learning: ResNet18 llega con pesos que ya detectan bordes, texturas y formas, mientras que la CNN custom debe aprender todo desde cero con un dataset nuevo.

Las clases con peor recall en ambos modelos (Bulldog, Cockapoo, razas Hairless)
corresponden a razas con características morfológicas similares, lo que sugiere que el límite está en la similitud intrínseca entre clases más que en la
capacidad del modelo.

## 8. Problemas encontrados y soluciones implementadas

- **Doble espacio en carpeta "American  Spaniel"**: el script de descarga generaba el
  nombre con dos espacios en la carpeta de validacion, causando que la raza no se encontrara al construir el dataset.
  Solución: renombrar con `mv "American  Spaniel" "American Spaniel"`.

- **Incompatibilidad de paths Windows/Linux en `index_image`**: `str(image_path)` en
  Windows genera rutas con barras invertidas (`\`) que luego son inválidas en Linux.
  Solución: reemplazar por `Path(image_path).as_posix()` en `SimilarityService.index_image`
  de `similarity_service.py`, que siempre produce barras normales (`/`).

## 9. Modificaciones fuera de las funciones indicadas

- **`src/lib/services/similarity_service.py` — `index_image`**: se reemplazó
  `path=str(image_path)` por `path=Path(image_path).as_posix()` para garantizar compatibilidad de rutas entre Windows y Linux.
